# Course-End Project: Building a RAG Pipeline Using PDF Chunking and Retrieval

**Objective:** Build a Retrieval-Augmented Generation (RAG) pipeline that loads PDF documents, chunks them, embeds the chunks with OpenAI's embedding model, stores them in a FAISS vectorstore, and retrieves the most relevant chunk(s) for a user query.

**Before running:** Drop your 2-3 PDF files into the `pdfs/` folder next to this notebook.

## Step 0: Setup and Imports

Make sure `OPENAI_API_KEY` is set in your environment (it already is in this workspace) and that `langchain`, `langchain-community`, `langchain-openai`, `pypdf`, and `faiss-cpu` are installed.

In [1]:
import os
import glob

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

# Confirm the API key is available (does not print the key itself)
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set in this environment."
print("OpenAI API key detected.")

PDF_FOLDER = "pdfs"
pdf_paths = sorted(glob.glob(os.path.join(PDF_FOLDER, "*.pdf")))
print(f"Found {len(pdf_paths)} PDF(s): {pdf_paths}")

C:\Users\Lawrence Miles III\AppData\Local\Temp\ipykernel_16608\26006808.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


OpenAI API key detected.
Found 3 PDF(s): ['pdfs\\NIST.AI.100-1.pdf', 'pdfs\\NIST.SP.1270.pdf', 'pdfs\\NIST.SP.800-218.pdf']


## Step 1: Load and Chunk PDF Content

Load each PDF with `PyPDFLoader`, then split the combined pages into overlapping chunks using `RecursiveCharacterTextSplitter`.

In [2]:
# Load all pages from all PDFs into a single list of Documents
all_pages = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    pages = loader.load()
    all_pages.extend(pages)
    print(f"Loaded {len(pages)} page(s) from {os.path.basename(path)}")

print(f"\nTotal pages loaded: {len(all_pages)}")

Loaded 48 page(s) from NIST.AI.100-1.pdf
Loaded 86 page(s) from NIST.SP.1270.pdf
Loaded 36 page(s) from NIST.SP.800-218.pdf

Total pages loaded: 170


In [3]:
# Chunk the loaded pages
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = text_splitter.split_documents(all_pages)
print(f"Total chunks created: {len(chunks)}")
print("\nSample chunk:\n" + "-" * 40)
print(chunks[0].page_content[:500])

Total chunks created: 577

Sample chunk:
----------------------------------------
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)


## Step 2: Generate Embeddings and Store in FAISS

Embed each chunk using OpenAI's embedding model and build a FAISS vectorstore for similarity search.

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(chunks, embeddings)

# Persist locally so we don't have to re-embed every run
vectorstore.save_local("faiss_index")
print("FAISS vectorstore built and saved to ./faiss_index")

FAISS vectorstore built and saved to ./faiss_index


## Step 3: Analyze the Chunking Results

Report the total number of chunks and the average chunk size (in characters), as required for the project report's Metrics section.

In [5]:
chunk_lengths = [len(c.page_content) for c in chunks]
total_chunks = len(chunks)
avg_chunk_size = sum(chunk_lengths) / total_chunks if total_chunks else 0

print(f"Total chunks: {total_chunks}")
print(f"Average chunk size: {avg_chunk_size:.1f} characters")
print(f"Min chunk size: {min(chunk_lengths)} | Max chunk size: {max(chunk_lengths)}")

Total chunks: 577
Average chunk size: 834.0 characters
Min chunk size: 76 | Max chunk size: 1000


## Step 4: Run a Retrieval Query

Accept a sample question, run semantic search against the FAISS vectorstore, and display the top matching chunk(s).

In [6]:
# TODO: replace with a question relevant to your own uploaded PDFs
sample_query = "What are some risks that can arise with improper use or software development?"

top_results = vectorstore.similarity_search(sample_query, k=3)

for i, result in enumerate(top_results, start=1):
    source = result.metadata.get("source", "unknown")
    page = result.metadata.get("page", "?")
    print(f"--- Match {i} (source: {os.path.basename(str(source))}, page: {page}) ---")
    print(result.page_content[:500])
    print()

--- Match 1 (source: NIST.AI.100-1.pdf, page: 12) ---
across other types of software development and deployment. Examples of overlapping risks
include: privacy concerns related to the use of underlying data to train AI systems; the en-
ergy and environmental implications associated with resource-heavy computing demands;
security concerns related to the confidentiality, integrity, and availability of the system and
its training and output data; and general security of the underlying software and hardware
for AI systems.
Page 8

--- Match 2 (source: NIST.AI.100-1.pdf, page: 9) ---
may complicate risk measurement. Risk can emerge both from third-party data, software or
hardware itself and how it is used. Risk metrics or methodologies used by the organization
developing the AI system may not align with the risk metrics or methodologies uses by
the organization deploying or operating the system. Also, the organization developing
the AI system may not be transparent about the risk metrics or

## Step 5 (Optional Enhancement): Full RAG Answer with GPT

Feed the retrieved chunks into a GPT chat completion so the assistant answers in natural language, grounded in your documents. This isn't strictly required by the rubric (which only asks for the top matching chunk) but strengthens the "AI-powered document assistant" framing from the assignment's Situation section.

In [7]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

context = "\n\n".join(r.page_content for r in top_results)
prompt = (
    "Answer the question using only the context below. "
    "If the answer isn't in the context, say you don't know.\n\n"
    f"Context:\n{context}\n\nQuestion: {sample_query}\nAnswer:"
)

response = llm.invoke(prompt)
print(response.content)

Some risks that can arise with improper use or software development include privacy concerns related to the use of underlying data to train AI systems, energy and environmental implications associated with resource-heavy computing demands, security concerns related to the confidentiality, integrity, and availability of the system and its training and output data, and general security of the underlying software and hardware for AI systems. Additionally, risks can emerge from third-party data, software, or hardware, and complications in risk measurement can occur if the risk metrics or methodologies used by different organizations do not align.


## Next Steps for Submission

1. Drop your 2-3 PDFs into the `pdfs/` folder and re-run all cells top to bottom.
2. Replace `sample_query` in Step 4 with 1-2 real questions about your documents and capture the outputs.
3. Write the short analysis report (Introduction, Implementation Overview, Metrics, Conclusion) using the printed chunk count, average chunk size, and sample query output from this notebook.
4. Zip this project folder (notebook + `pdfs/` + report) for LMS submission.